In [ ]:
import os
import cv2
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, classification_report, f1_score, roc_auc_score
from skimage.feature import hog

TRAIN_DIR = "data/agyikepek_4_osztaly/Training"
TEST_DIR = "data/agyikepek_4_osztaly/Testing"
IMG_SIZE = (128, 128)

CATEGORIES = ["glioma", "meningioma", "notumor", "pituitary"]

In [ ]:
# Lassu betoltes
def preprocess_and_segment(image_path):
    """
    Hagyományos képfeldolgozás és alapfokú szegmentáció (háttér-maszkolás)
    """
    img = cv2.imread(image_path, cv2.IMREAD_GRAYSCALE)
    if img is None:
        return None

    # 1. Zajszűrés Gauss-szűrővel
    blurred = cv2.GaussianBlur(img, (5, 5), 0)
    
    # 2. SZEGMENTÁCIÓ: Otsu-féle automatikus küszöbölés a háttér elkülönítésére
    _, mask = cv2.threshold(blurred, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    
    # Maszk alkalmazása: csak az agyszövet marad meg, a külső zajok eltűnnek
    segmented_img = cv2.bitwise_and(img, img, mask=mask)
    
    # 3. Kontraszt növelés (CLAHE) a szegmentált régión
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    enhanced_img = clahe.apply(segmented_img)
    
    # 4. Átméretezés a fix jellemzővektor mérethez
    resized_img = cv2.resize(enhanced_img, IMG_SIZE)
    
    # 5. JELLEMZŐKINYERÉS: HOG (Histogram of Oriented Gradients)
    features = hog(
        resized_img, 
        orientations=9, 
        pixels_per_cell=(16, 16), 
        cells_per_block=(2, 2), 
        visualize=False
    )
    
    # Alapvető statisztikai jellemzők hozzáadása (intenzitás eloszlás)
    mean_val = np.mean(resized_img)
    std_val = np.std(resized_img)
    final_features = np.append(features, [mean_val, std_val])
    
    return final_features

def load_dataset(base_dir):
    X, y = [], []
    for category_idx, category in enumerate(CATEGORIES):
        folder_path = os.path.join(base_dir, category)
        if not os.path.exists(folder_path):
            continue
        print(f" Feldolgozás: {category}...")
        for filename in os.listdir(folder_path):
            if filename.lower().endswith(('.png', '.jpg', '.jpeg')):
                img_path = os.path.join(folder_path, filename)
                features = preprocess_and_segment(img_path)
                if features is not None:
                    X.append(features)
                    y.append(category_idx)
    return np.array(X), np.array(y)

# --- ADATOK BETÖLTÉSE ---
print("--- 1. TANÍTÓ HALMAZ BETÖLTÉSE ---")
X_train, y_train = load_dataset(TRAIN_DIR)

print("\n--- 2. TESZTELŐ HALMAZ BETÖLTÉSE ---")
X_test, y_test = load_dataset(TEST_DIR)

In [5]:
#Gyorsabb betöltés párhuzamosítással (joblib)
from joblib import Parallel, delayed

def process_single_image(filename, folder_path, category_idx):
    """Különálló függvény egyetlen kép feldolgozására (ezt futtatjuk majd a magokon)"""
    if filename.lower().endswith(('.png', '.jpg', '.jpeg')):
        img_path = os.path.join(folder_path, filename)
        features = preprocess_and_segment(img_path)
        if features is not None:
            return features, category_idx
    return None

def load_dataset_fast(base_dir):
    if not os.path.exists(base_dir):
        print(f"❌ HIBA: A fő mappa NEM LÉTEZIK: {base_dir}")
        return np.array([]), np.array([])
        
    # 1. Összegyűjtjük az összes feladatot egy listába
    tasks = []
    for category_idx, category in enumerate(CATEGORIES):
        folder_path = os.path.join(base_dir, category)
        if not os.path.exists(folder_path):
            continue
        for filename in os.listdir(folder_path):
            tasks.append((filename, folder_path, category_idx))
            
    print(f"⚡ Összegyűjtve: {len(tasks)} kép. Párhuzamos feldolgozás indítása az ÖSSZES CPU magon...")
    
    # 2. Lefuttatjuk párhuzamosan az összes elérhető CPU magon (n_jobs=-1)
    # A backend='loky' a legstabilabb dolog Windows alatt
    results = Parallel(n_jobs=-1, backend='loky')(
        delayed(process_single_image)(f, fp, idx) for f, fp, idx in tasks
    )
    
    # 3. Az eredmények szétválogatása
    X, y = [], []
    for res in results:
        if res is not None:
            X.append(res[0])
            y.append(res[1])
            
    return np.array(X), np.array(y)

# --- ADATOK BETÖLTÉSE (GYORSÍTOTT VERZIÓ) ---
print("--- 1. TANÍTÓ HALMAZ BETÖLTÉSE ---")
X_train, y_train = load_dataset_fast(TRAIN_DIR)
print(f"✅ Sikeresen betöltve a tanító halmaz: {len(X_train)} kép.")

print("\n--- 2. TESZTELŐ HALMAZ BETÖLTÉSE ---")
X_test, y_test = load_dataset_fast(TEST_DIR)
print(f"✅ Sikeresen betöltve a tesztelő halmaz: {len(X_test)} kép.")

# Biztonsági ellenőrzés
if len(X_train) == 0 or len(X_test) == 0:
    raise ValueError("❌ Valami hiba van, az adathalmaz üres maradt!")

--- 1. TANÍTÓ HALMAZ BETÖLTÉSE ---
⚡ Összegyűjtve: 5599 kép. Párhuzamos feldolgozás indítása az ÖSSZES CPU magon...
✅ Sikeresen betöltve a tanító halmaz: 5599 kép.

--- 2. TESZTELŐ HALMAZ BETÖLTÉSE ---
⚡ Összegyűjtve: 1311 kép. Párhuzamos feldolgozás indítása az ÖSSZES CPU magon...
✅ Sikeresen betöltve a tesztelő halmaz: 1311 kép.


In [ ]:
from sklearn.metrics import accuracy_score, classification_report, f1_score, roc_auc_score

# --- MODELL COMPASS (TÖBB MODELL ÖSSZEHASONLÍTÁSA) ---
# Itt adtuk hozzá a probability=True paramétert az SVM-hez!
models = {
    "Random Forest": RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1),
    "Support Vector Machine (SVM)": SVC(kernel='rbf', C=1.0, probability=True, random_state=42),
    "K-Nearest Neighbors (KNN)": KNeighborsClassifier(n_neighbors=5, n_jobs=-1)
}

results_data = []

print("\n--- 3. MODELLEK TANÍTÁSA ÉS KIÉRTÉKELÉSE ---")
for name, model in models.items():
    print(f"{name} tanítása folyamatban...")
    model.fit(X_train, y_train)
    
    # Predikciók a metrikákhoz
    y_pred = model.predict(X_test)
    y_probs = model.predict_proba(X_test)  # Most már hibamentesen lefut!
    
    # Alapmetrikák kiszámítása
    acc = accuracy_score(y_test, y_pred)
    macro_f1 = f1_score(y_test, y_pred, average='macro')
    
    # Többosztályos ROC-AUC számítás (One-vs-Rest módszerrel)
    roc_auc = roc_auc_score(y_test, y_probs, multi_class='ovr', average='macro')
    
    # Mentés a végső táblázathoz
    results_data.append({
        "Modell": name, 
        "Accuracy": round(acc, 4),
        "Macro F1-Score": round(macro_f1, 4),
        "ROC-AUC (OvR)": round(roc_auc, 4)
    })
    
    print(f"-> {name} kész.")
    print(classification_report(y_test, y_pred, target_names=CATEGORIES))
    print("-" * 50)

# --- VÉGSŐ ÖSSZEHASONLÍTÓ TÁBLÁZAT ---
print("\n--- VÉGSŐ ÖSSZEHASONLÍTÓ ÖSSZEGZÉS ---")
df_results = pd.DataFrame(results_data)
print(df_results.to_string(index=False))


--- 3. MODELLEK TANÍTÁSA ÉS KIÉRTÉKELÉSE ---
Random Forest tanítása folyamatban...
-> Random Forest kész.
              precision    recall  f1-score   support

      glioma       0.87      0.86      0.86       300
  meningioma       0.85      0.82      0.83       306
     notumor       0.98      0.99      0.98       405
   pituitary       0.93      0.96      0.94       300

    accuracy                           0.91      1311
   macro avg       0.91      0.91      0.91      1311
weighted avg       0.91      0.91      0.91      1311

--------------------------------------------------
Support Vector Machine (SVM) tanítása folyamatban...
